In [2]:
import pandas as pd
import numpy as np

np.random.seed(42)

# Create a 6-month hourly date range
dates = pd.date_range(start="2024-01-01", end="2024-06-30", freq="H")

# 5 companies
companies = ["AAPL", "GOOG", "AMZN", "MSFT", "NVDA"]

# Build large DataFrame
df = pd.DataFrame({
    "Timestamp": np.repeat(dates, len(companies)),
    "Company": np.tile(companies, len(dates)),
})

# Simulate prices (lognormal random walk)
df["Price"] = (
    100
    * np.exp(np.cumsum(np.random.normal(0, 0.002, len(df))) / 10)
).round(2)

df.head()


C:\Users\zesty\AppData\Local\Temp\ipykernel_23592\2207889483.py:7: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  dates = pd.date_range(start="2024-01-01", end="2024-06-30", freq="H")


,Timestamp,Company,Price
0,2024-01-01,AAPL,100.01
1,2024-01-01,GOOG,100.01
2,2024-01-01,AMZN,100.02
3,2024-01-01,MSFT,100.05
4,2024-01-01,NVDA,100.05


In [3]:
df['Return'] = df.groupby('Company')['Price'].pct_change()

In [4]:
df

,Timestamp,Company,Price,Return
0,2024-01-01,AAPL,100.01,NaN
1,2024-01-01,GOOG,100.01,NaN
2,2024-01-01,AMZN,100.02,NaN
3,2024-01-01,MSFT,100.05,NaN
4,2024-01-01,NVDA,100.05,NaN
...,...,...,...,...
21720,2024-06-30,AAPL,101.93,-0.000490
21721,2024-06-30,GOOG,101.93,-0.000490
21722,2024-06-30,AMZN,101.94,0.000000
21723,2024-06-30,MSFT,101.94,0.000196


In [5]:
df['hour'] = df['Timestamp'].dt.hour
df['date'] = df['Timestamp'].dt.date
avg_price = df.groupby(['Company', 'date'])['Price'].mean().reset_index()
std_price = df.groupby(['Company'])['Return'].std().reset_index()

In [6]:
df

,Timestamp,Company,Price,Return,hour,date
0,2024-01-01,AAPL,100.01,NaN,0,2024-01-01
1,2024-01-01,GOOG,100.01,NaN,0,2024-01-01
2,2024-01-01,AMZN,100.02,NaN,0,2024-01-01
3,2024-01-01,MSFT,100.05,NaN,0,2024-01-01
4,2024-01-01,NVDA,100.05,NaN,0,2024-01-01
...,...,...,...,...,...,...
21720,2024-06-30,AAPL,101.93,-0.000490,0,2024-06-30
21721,2024-06-30,GOOG,101.93,-0.000490,0,2024-06-30
21722,2024-06-30,AMZN,101.94,0.000000,0,2024-06-30
21723,2024-06-30,MSFT,101.94,0.000196,0,2024-06-30


In [7]:
avg_price

,Company,date,Price
0,AAPL,2024-01-01,99.854167
1,AAPL,2024-01-02,99.871250
2,AAPL,2024-01-03,99.996250
3,AAPL,2024-01-04,100.139583
4,AAPL,2024-01-05,99.935833
...,...,...,...
905,NVDA,2024-06-26,101.688750
906,NVDA,2024-06-27,101.787500
907,NVDA,2024-06-28,101.757083
908,NVDA,2024-06-29,101.894167


In [8]:
std_price

,Company,Return
0,AAPL,0.000439
1,AMZN,0.000446
2,GOOG,0.000441
3,MSFT,0.000451
4,NVDA,0.000446


In [9]:
df = df.merge(avg_price, on = ['Company', 'date'])
df = df.merge(std_price, on = ['Company'])

In [10]:
df['Extreme'] = np.abs(df['Return_x']) > 2*df['Return_y']

In [11]:
extremes = df.loc[df["Extreme"]]
extremes = extremes.groupby('Company')['Timestamp'].diff()

In [12]:
# Filter for extreme moves only
extremes = df.loc[df["Extreme"]]

# Compute time since last extreme move per company
extremes["TimeSinceLast"] = extremes.groupby("Company")["Timestamp"].diff()


C:\Users\zesty\AppData\Local\Temp\ipykernel_23592\2291826380.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  extremes["TimeSinceLast"] = extremes.groupby("Company")["Timestamp"].diff()


In [13]:
extremes = df.loc[df["Extreme"]]
extremes["TimeSinceLast"] = extremes.groupby('Company')['Timestamp'].diff()

C:\Users\zesty\AppData\Local\Temp\ipykernel_23592\2185646260.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  extremes["TimeSinceLast"] = extremes.groupby('Company')['Timestamp'].diff()


In [14]:
df['rolling'] = df.groupby('Company')['Return_x'].transform(lambda x: x.rolling(window = 10).mean())